# Day 04 — Embeddings, lexical matching and similarity: two ways to decide that a chunk matches

*Part of **15 Days of Modern, Enterprise-Relevant RAG from Scratch**. Lesson: [`docs/day-04/LEARN.md`](../docs/day-04/LEARN.md).*

Day 01's BM25 retriever matches **words**. Ask it *"How much time off do new dads get?"* and it can't find the parental leave policy, because the policy never says "dads": it says "primary caregivers" and "secondary caregivers". An **embedding model** turns text into a list of numbers, so that texts with similar meaning get similar numbers whatever words they use. That fixes some failures and brings new ones.

Today both retrievers get exactly the same input and the same questions:

```
 44 chunks from Day 03 ──┬──> BM25 (Day 01): which question words does the chunk contain, and how rare are they?
 ("Title > Section"      │
  header, ≤ 80 words)    └──> embeddings: chunk -> 384 numbers, question -> 384 numbers, cosine similarity
            both ──> top 3 for the same questions ──> where does each one find the evidence, and where does it fail?
```

| This notebook **does** | This notebook **does not** |
|---|---|
| Download a small open embedding model at a pinned revision and check every file's SHA-256 | Use a vector database or an approximate index (Day 05) |
| Run it on the CPU and show every step: tokens, vectors, pooling, normalisation | Combine the two retrievers into one ranking (Day 08) |
| Explain cosine similarity, dot product and distance, and when they disagree | Generate answers with an LLM (Day 06) |
| Compare BM25 and embeddings on 17 questions (Day 03's 13, Day 01's 4 hardest) and 30 exact lookups | Declare a winner: these questions are a diagnostic, not a benchmark (Day 07) |
| Re-measure Day 03's context headers and the model's query instruction | Call a paid API or send your text to anyone |
| Show what to store next to every vector so updates and model changes stay safe | Enforce permissions (Days 05 and 14), although every chunk still carries them |

**Cost:** free. One-time download of the model (133 MB) from Hugging Face; after that it runs offline. **Runtime:** under a minute after the download. **Requirements:** Python 3.11+ and `requirements.txt`; Day 04 adds `numpy`, `onnxruntime`, `tokenizers` and `huggingface-hub` (about 50 MB to download). No GPU, no API key.
**How to run:** open this notebook from inside the cloned repository, then *Kernel → Restart Kernel and Run All Cells*.
**Verified:** executed top to bottom on 2026-10-16 on Linux with Python 3.11, 3.12, 3.13 and 3.14. The saved outputs come from the Python 3.13 run. The model's first download from Hugging Face was not run in that environment (its files came from a copy matching the published SHA-256 values); CI runs the download and the notebook on Linux, Windows and macOS on every push.

## 0. Setup

Two kinds of imports: the new packages that download and run the embedding model, and the shared code from earlier days in [`src/rag_from_scratch/`](../src/rag_from_scratch/): Day 02's ingestion, Day 01's BM25 and, new today, **Day 03's structure-aware chunker and its 13 questions**. `tests/test_shared_code.py` checks that each shared copy returns exactly what its original notebook returns.

In [1]:
import hashlib
import json
import platform
import re
import statistics
import sys
import time
import warnings
from dataclasses import dataclass, replace
from pathlib import Path

assert sys.version_info >= (3, 11), "This series needs Python 3.11 or newer."
# huggingface_hub's progress bars warn when Jupyter widgets aren't installed; they still work as plain text.
warnings.filterwarnings("ignore", message="IProgress not found")

try:
    import numpy as np
    import onnxruntime as ort
    import tokenizers
    from huggingface_hub import hf_hub_download
    from tokenizers import Tokenizer
except ImportError as error:
    raise ImportError("Day 04 needs numpy, onnxruntime, tokenizers and huggingface-hub. From the repository "
                      "root run: python -m pip install -r requirements.txt") from error


def find_repo_root() -> Path:
    # Walk up until the shared package is found, so the notebook works from the repo root or
    # from notebooks/. No hard-coded paths.
    here = Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "src" / "rag_from_scratch").is_dir():
            return folder
    raise FileNotFoundError("Could not find src/rag_from_scratch. Start Jupyter from inside the cloned repository.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))           # import the shared code without installing it

from rag_from_scratch.bm25 import BM25Index, tokenize                # Day 01's retriever
from rag_from_scratch.chunking import count_words, section_chunks   # Day 03's chunker
from rag_from_scratch.ingest import ingest                           # Day 02's pipeline
from rag_from_scratch.probes import DAY03_PROBES, Probe              # Day 03's 13 questions

print(f"Python {platform.python_version()} | numpy {np.__version__} | onnxruntime {ort.__version__} "
      f"| tokenizers {tokenizers.__version__}")

Python 3.13.16 | numpy 2.4.6 | onnxruntime 1.31.0 | tokenizers 0.23.3


In [2]:
TOP_K = 3   # chunks retrieved per question, as on Days 01 and 03

MODEL = {
    "repo": "BAAI/bge-small-en-v1.5",                       # https://huggingface.co/BAAI/bge-small-en-v1.5
    "revision": "5c38ec7c405ec4b44b94cc5a9bb96e735b38267a",  # a commit, not "main": the files can't change
    "licence": "MIT",
    "files": {                                              # path in the repository: (SHA-256, bytes)
        "onnx/model.onnx": ("828e1496d7fabb79cfa4dcd84fa38625c0d3d21da474a00f08db0f559940cf35", 133_093_490),
        "tokenizer.json": ("d241a60d5e8f04cc1b2b3e9ef7a4921b27bf526d9f6050ab90f9267a1f9e5c66", 711_396),
    },
}
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "   # model card: queries only
MAX_TOKENS = 512    # the model's input limit; longer texts are cut off (our chunks are far shorter)

## 1. The input — Day 03's chunks

Day 03 showed that cutting along the document's structure keeps answers whole. Its chunker now lives in `src/rag_from_scratch/chunking.py`, so this notebook rebuilds exactly the same 44 chunks instead of reading Day 03's output file. It works even if you never ran Day 03.

In [3]:
documents, quarantined = ingest(REPO_ROOT)
chunks = [c for d in documents for c in section_chunks(d)]
by_id = {c.chunk_id: c for c in chunks}
chunk_ids = [c.chunk_id for c in chunks]

sizes = [count_words(c.text) for c in chunks]
print(f"{len(documents)} documents -> {len(chunks)} chunks, {min(sizes)}-{max(sizes)} words each "
      f"(median {statistics.median(sizes):.0f})\n")
print(by_id["parental-leave-policy#leave-entitlement"].text)

11 documents -> 44 chunks, 18-74 words each (median 34)

Parental Leave Policy > Leave entitlement
Primary caregivers receive 16 weeks of fully paid leave. Secondary caregivers receive 6 weeks of fully paid leave. Leave can be taken in up to two blocks within 12 months of the child's arrival.


That chunk answers the "new dads" question: a new father is a primary or a secondary caregiver. But the question and the chunk share no useful word, which is exactly where BM25 is helpless and embeddings are supposed to help.

## 2. Get the model — a pinned revision, checked by hash

**Which model, and why.** [`BAAI/bge-small-en-v1.5`](https://huggingface.co/BAAI/bge-small-en-v1.5) is a small English embedding model (33 million parameters, 384 numbers per text) released in September 2023 under the MIT licence. It runs comfortably on a laptop CPU, and its official repository includes an **ONNX** file: a portable model format that the `onnxruntime` library can run without PyTorch, a much larger install. Newer and stronger models exist (LEARN.md explains how to choose one); everything below works the same with any of them.

**Treat a model file like a dependency.** It decides every result in this notebook, so we pin it like a package:

1. **A commit, not "main".** We ask for an exact revision of the model repository. If the authors update the model, our results don't silently change.
2. **A SHA-256 per file.** The notebook hashes each file and refuses to continue if a hash differs: that catches a broken download, a wrong mirror and a tampered file.
3. **A safe format.** ONNX and `tokenizer.json` are data. Some model formats (Python pickles such as `pytorch_model.bin`) can run code when they are loaded; don't load those from sources you don't trust.

The first run downloads the two files with `hf_hub_download` from Hugging Face's official `huggingface_hub` library, into `models/` at the repository root (ignored by Git). It uses your operating system's certificate store and the usual `HTTPS_PROXY` setting. Once the files are there, the notebook doesn't call the library again, so later runs make no network requests at all. If your network blocks Hugging Face, set `HF_ENDPOINT` to your organisation's mirror, or download the two files another way and save them where the error message says: the hash check still applies.

In [4]:
MODEL_DIR = REPO_ROOT / "models" / "bge-small-en-v1.5" / MODEL["revision"][:12]   # models/ is git-ignored


def file_sha256(path: Path) -> str:
    with path.open("rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


def model_file(name: str) -> Path:
    # Download one file of the pinned revision if it isn't here yet, then check its SHA-256 every time.
    expected_sha256, expected_size = MODEL["files"][name]
    local = MODEL_DIR / name
    shown = local.relative_to(REPO_ROOT).as_posix()
    if not local.is_file():   # only the first run touches the network
        try:
            hf_hub_download(MODEL["repo"], name, revision=MODEL["revision"], local_dir=MODEL_DIR)
        except Exception as error:   # no network, a proxy, a blocked host...: say how to fix it
            raise RuntimeError(f"Could not download {name} ({expected_size / 1e6:.1f} MB) from {MODEL['repo']}: "
                               f"{error}\nBehind a proxy? Set HTTPS_PROXY. Hugging Face blocked? Set HF_ENDPOINT to a "
                               f"mirror, or download the file yourself and save it as {shown}") from error
    actual = file_sha256(local)
    if actual != expected_sha256:
        raise ValueError(f"{shown} has SHA-256 {actual}, expected {expected_sha256}. "
                         "Delete it and run again. Never load a model file you can't verify.")
    print(f"{name:<16} {local.stat().st_size:>11,} bytes   SHA-256 {actual[:12]}... verified")
    return local


tokenizer_path = model_file("tokenizer.json")
model_path = model_file("onnx/model.onnx")

tokenizer.json       711,396 bytes   SHA-256 d241a60d5e8f... verified


onnx/model.onnx  133,093,490 bytes   SHA-256 828e1496d7fa... verified


In [5]:
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.enable_truncation(max_length=MAX_TOKENS)
tokenizer.enable_padding()      # in a batch, pad shorter texts with [PAD] up to the longest one

session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
INPUT_NAMES = [i.name for i in session.get_inputs()]

print(f"tokenizer: {tokenizer.get_vocab_size():,} entries in the vocabulary")
print(f"model inputs:  {INPUT_NAMES}")
print(f"model outputs: {[(o.name, o.shape) for o in session.get_outputs()]}")

tokenizer: 30,522 entries in the vocabulary
model inputs:  ['input_ids', 'attention_mask', 'token_type_ids']
model outputs: [('last_hidden_state', ['batch_size', 'sequence_length', 384])]


## 3. From text to vector, step by step

An embedding model is a function from text to a fixed-length list of numbers, a **vector**. Four steps happen inside. We run each one by hand once, then wrap them in one `embed()` function.

### 3.1 Tokens

The model doesn't read words; it reads **tokens** from a fixed vocabulary of 30,522 entries. A word it doesn't know whole is split into known pieces (`##` marks a piece that continues a word). Compare with what BM25 sees for the same question:

In [6]:
QUESTION = "How much time off do new dads get?"
encoding = tokenizer.encode(QUESTION)

print("model tokens:", encoding.tokens)
print("token IDs:   ", encoding.ids)
print("BM25 terms:  ", tokenize(QUESTION))

model tokens: ['[CLS]', 'how', 'much', 'time', 'off', 'do', 'new', 'dad', '##s', 'get', '?', '[SEP]']
token IDs:    [101, 2129, 2172, 2051, 2125, 2079, 2047, 3611, 2015, 2131, 1029, 102]
BM25 terms:   ['much', 'time', 'off', 'new', 'dads', 'get']


BM25 drops stop words ("how", "do") and keeps whole words: `dads` either appears in a chunk or it doesn't. The model keeps every token, punctuation included, adds two markers (`[CLS]` at the start, `[SEP]` at the end) and splits "dads" into `dad` + `##s`. Nothing in this step knows that a dad can be a caregiver. That knowledge is in the model's weights, learned from large amounts of text.

### 3.2 The model: one vector per token

The model reads all tokens together and returns one vector of 384 numbers **per token**, each one informed by the whole text:

In [7]:
def model_inputs(encodings) -> dict[str, np.ndarray]:
    # Tokenizer output -> the three integer arrays the model expects, one row per text.
    return {
        "input_ids": np.array([e.ids for e in encodings], dtype=np.int64),
        "attention_mask": np.array([e.attention_mask for e in encodings], dtype=np.int64),
        "token_type_ids": np.array([e.type_ids for e in encodings], dtype=np.int64),
    }


def run_model(inputs: dict[str, np.ndarray]) -> np.ndarray:
    # Token IDs in, one 384-number vector per token out: shape (texts, tokens, 384).
    return session.run(None, {name: inputs[name] for name in INPUT_NAMES})[0]


token_vectors = run_model(model_inputs([encoding]))
print(f"shape {token_vectors.shape}: {token_vectors.shape[0]} text, {token_vectors.shape[1]} tokens, "
      f"{token_vectors.shape[2]} numbers per token")

shape (1, 12, 384): 1 text, 12 tokens, 384 numbers per token


### 3.3 Pooling: one vector for the whole text

Retrieval needs **one** vector per text, so the token vectors are combined. This is called **pooling**. Each model is trained with one pooling method, and using another one gives worse results without any error message. This model uses the vector of the first token, `[CLS]` (its model card and its `1_Pooling/config.json` say so). Many other models average all the token vectors instead (*mean pooling*). Check the model card every time.

### 3.4 Normalisation: length 1

Finally the vector is divided by its length, so that every vector has length 1. Section 4 shows why that matters.

In [8]:
cls_vector = token_vectors[0, 0]                      # position 0 is the [CLS] token
unit_vector = cls_vector / np.linalg.norm(cls_vector)

print(f"[CLS] vector: {len(cls_vector)} numbers, length {np.linalg.norm(cls_vector):.2f}")
print(f"normalised:   length {np.linalg.norm(unit_vector):.2f}, first six numbers {np.round(unit_vector[:6], 3)}")

[CLS] vector: 384 numbers, length 9.17
normalised:   length 1.00, first six numbers [-0.015  0.044  0.01  -0.059  0.01   0.026]


The individual numbers mean nothing on their own; only comparisons between vectors do. Now the four steps as one function. It embeds texts in small batches, which is faster, and that brings one detail: in a batch, shorter texts are padded with `[PAD]` tokens, and the **attention mask** (1 = real token, 0 = padding) tells the model to ignore them.

In [9]:
def embed(texts: list[str], batch_size: int = 16) -> np.ndarray:
    # text -> tokens -> model -> [CLS] vector -> length 1. One row per text, 384 columns.
    rows = []
    for start in range(0, len(texts), batch_size):
        token_vectors = run_model(model_inputs(tokenizer.encode_batch(texts[start:start + batch_size])))
        cls = token_vectors[:, 0, :]
        rows.append(cls / np.linalg.norm(cls, axis=1, keepdims=True))
    return np.vstack(rows).astype(np.float32)


def embed_query(question: str, instruction: str = QUERY_INSTRUCTION) -> np.ndarray:
    # The model card asks for this instruction in front of short search queries (not in front of passages).
    return embed([instruction + question])[0]


# Check: padding must not change a vector. Embed the question alone, then in a batch with a longer chunk.
LONGER = by_id["travel-policy#daily-limits-by-city"].text
alone = embed([QUESTION])[0]
in_batch = embed([QUESTION, LONGER])[0]
batch_inputs = model_inputs(tokenizer.encode_batch([QUESTION, LONGER]))
print(f"in a batch with the travel-limits chunk, the question gets "
      f"{(batch_inputs['attention_mask'][0] == 0).sum()} [PAD] tokens")
print(f"alone vs in the batch:               largest difference {np.abs(alone - in_batch).max():.1e}")

# What happens if the model is NOT told which tokens are padding?
batch_inputs["attention_mask"][:] = 1                  # pretend the [PAD] tokens are real text
unmasked = run_model(batch_inputs)[0, 0]
unmasked = unmasked / np.linalg.norm(unmasked)
print(f"alone vs the batch without the mask: largest difference {np.abs(alone - unmasked).max():.1e}, "
      f"cosine similarity {alone @ unmasked:.2f}")

in a batch with the travel-limits chunk, the question gets 71 [PAD] tokens
alone vs in the batch:               largest difference 0.0e+00


alone vs the batch without the mask: largest difference 1.3e-01, cosine similarity 0.70


With the mask, the question's vector is the same alone or in a batch. Without it, the model reads 71 padding tokens as part of the question and returns a clearly different vector (cosine 0.70 with the correct one). Libraries do this for you; it is worth knowing when you write your own batching code, or when a library's results change with the batch size.

## 4. Similarity: cosine, dot product and distance

Three common ways to compare two vectors **a** and **b**:

| Measure | Formula | Higher means |
|---|---|---|
| **Dot product** | a · b = Σ aᵢbᵢ | more similar |
| **Cosine similarity** | (a · b) / (‖a‖ ‖b‖), the cosine of the angle between them | more similar (range −1 to 1) |
| **Euclidean distance** | ‖a − b‖ | *less* similar |

For vectors of length 1 they all agree: cosine equals the dot product, and distance² = 2 − 2 × cosine, so all three rank chunks identically. That is why we normalised.

In [10]:
a, b = embed([QUESTION, by_id["parental-leave-policy#leave-entitlement"].text])
cosine = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"cosine {cosine:.4f} | dot product {a @ b:.4f} | distance² {np.linalg.norm(a - b) ** 2:.4f} "
      f"| 2 - 2 × cosine {2 - 2 * cosine:.4f}")

cosine 0.6849 | dot product 0.6849 | distance² 0.6301 | 2 - 2 × cosine 0.6301


**When they disagree.** If vectors are *not* normalised, the dot product rewards long vectors. A two-number toy example:

In [11]:
query = np.array([1.0, 1.0])
candidates = {"same direction, short": np.array([1.0, 0.9]), "other direction, long": np.array([3.0, 0.0])}
for name, v in candidates.items():
    cos = query @ v / (np.linalg.norm(query) * np.linalg.norm(v))
    print(f"{name:<22} dot product {query @ v:4.2f}   cosine {cos:4.2f}")

same direction, short  dot product 1.90   cosine 1.00
other direction, long  dot product 3.00   cosine 0.71


The dot product picks the long vector; cosine picks the one pointing the same way. Neither is "right" in general. Some models are trained so that a vector's length carries information, and should be compared with the dot product; others, like this one, are trained for cosine similarity on normalised vectors. **Use the measure the model was trained with** (the model card says), and never compare vectors from different models: their numbers live in different spaces.

**Scores are not percentages.** Here are the cosine similarities between five short texts, most of them about unrelated things:

In [12]:
SENTENCES = [
    "How much time off do new dads get?",
    "Secondary caregivers receive 6 weeks of fully paid leave.",
    "Your account locks after five failed sign-in attempts.",
    "What is the hotel limit per night in Singapore?",
    "What is the capital of France?",
]
vectors = embed(SENTENCES)
similarity = vectors @ vectors.T
print(" " * 60 + "".join(f"{i:>7}" for i in range(1, len(SENTENCES) + 1)))
for i, (sentence, row) in enumerate(zip(SENTENCES, similarity), start=1):
    print(f"{i} {sentence:<58}" + "".join(f"{s:7.2f}" for s in row))

unrelated = [similarity[i, j] for i in range(5) for j in range(i + 1, 5) if (i, j) != (0, 1)]
print(f"\nthe one related pair (1 and 2): {similarity[0, 1]:.2f}; the nine other pairs: "
      f"{min(unrelated):.2f} to {max(unrelated):.2f}")

                                                                  1      2      3      4      5
1 How much time off do new dads get?                           1.00   0.62   0.46   0.46   0.30
2 Secondary caregivers receive 6 weeks of fully paid leave.    0.62   1.00   0.46   0.50   0.35
3 Your account locks after five failed sign-in attempts.       0.46   0.46   1.00   0.44   0.34
4 What is the hotel limit per night in Singapore?              0.46   0.50   0.44   1.00   0.39
5 What is the capital of France?                               0.30   0.35   0.34   0.39   1.00

the one related pair (1 and 2): 0.62; the nine other pairs: 0.30 to 0.50


The question about new dads and the sentence about caregivers score 0.62: the highest in the table, so the model does connect them. But texts about completely different things still score 0.30 to 0.50, never anywhere near 0. With this model, a similarity is only meaningful **compared with other similarities for the same question**. The model card says the same: use the order of the scores, and if you need a threshold, choose it on your own data. Section 7.2 tries.

## 5. Dense retrieval in a few lines

"Dense" because every text becomes a vector in which every number is used. BM25's view of a text is *sparse*: a handful of words out of a large vocabulary.

- **Index time:** embed every chunk once and keep the vectors: a table of 44 rows and 384 columns.
- **Query time:** embed the question, compute its cosine with every chunk vector (one matrix multiplication, because all vectors have length 1), return the top 3.

In [13]:
started = time.perf_counter()
chunk_vectors = embed([c.text for c in chunks])
index_seconds = time.perf_counter() - started
print(f"embedded {len(chunks)} chunks in {index_seconds:.1f} s -> matrix {chunk_vectors.shape}, "
      f"{chunk_vectors.nbytes:,} bytes as float32\n")


@dataclass(frozen=True)
class Match:
    rank: int
    score: float
    id: str


def dense_search(question: str, vectors: np.ndarray | None = None, k: int = TOP_K,
                 instruction: str = QUERY_INSTRUCTION) -> list[Match]:
    vectors = chunk_vectors if vectors is None else vectors
    scores = vectors @ embed_query(question, instruction)      # cosine similarity with every chunk
    best = np.argsort(-scores, kind="stable")[:k]
    return [Match(rank, float(scores[i]), chunk_ids[i]) for rank, i in enumerate(best, start=1)]


bm25 = BM25Index({c.chunk_id: c.text for c in chunks})          # Day 01's retriever on the same chunks


def show_hits(question: str) -> None:
    print(question)
    for name, hits in [("BM25", bm25.search(question, k=TOP_K)), ("embeddings", dense_search(question))]:
        print(f"  {name}")
        if not hits:
            print("    (nothing: no chunk shares a word with the question)")
        for hit in hits:
            text = by_id[hit.id].text.split("\n", 1)[1].replace("\n", " / ")
            print(f"    #{hit.rank} {hit.score:5.2f}  [{hit.id}]  {text[:56]}{'...' if len(text) > 56 else ''}")


show_hits(QUESTION)

embedded 44 chunks in 1.1 s -> matrix (44, 384), 67,584 bytes as float32

How much time off do new dads get?
  BM25
    #1  3.51  [security-incident-reporting#what-not-to-do]  Do not try to investigate the incident yourself, delete ...
    #2  2.89  [remote-work-policy#purpose]  This policy explains how Acme employees can work remotel...
    #3  2.78  [new-starter-it-checklist#your-first-day]  Collect your laptop and security badge from reception. S...
  embeddings
    #1  0.72  [parental-leave-policy#returning-to-work]  You may return on a phased schedule of reduced hours for...
    #2  0.67  [parental-leave-policy#who-is-eligible]  All employees with at least six months of service are el...
    #3  0.66  [parental-leave-policy#leave-entitlement]  Primary caregivers receive 16 weeks of fully paid leave....


BM25's top 3 have nothing to do with parental leave. Embeddings return three chunks of the parental leave policy, the right document, but the chunk that answers the question (`leave-entitlement`) is third, behind *Returning to work* and *Who is eligible*. Better, not solved.

Note the two score columns. BM25 scores and cosine similarities are on completely different scales, so they can't be compared or added up. (Day 08 combines the two retrievers by rank instead of by score.)

## 6. Head to head: 17 questions

The same evidence check as Day 03: a retrieved chunk counts only if it contains **all** of a question's evidence strings, so that it can answer on its own. We use Day 03's 13 questions and add four from Day 01's list of failures:

- two whose answer is in the documents in different words (*new dads* vs *caregivers*; *locked* vs *locks*);
- two that the documents **cannot** answer. The right result is no evidence at all. Their evidence is empty, and the table shows the top score instead.

In [14]:
DAY01_QUESTIONS = (
    Probe("How much time off do new dads get?",
          ("Primary caregivers receive 16 weeks", "Secondary caregivers receive 6 weeks"), "Day 01: different words"),
    Probe("My account is locked, what now?", ("It unlocks automatically after 15 minutes",),
          "Day 01: locked vs locks"),
    Probe("What is Google's remote work policy?", (), "Day 01: out of scope, on-topic words"),
    Probe("What is the capital of France?", (), "Day 01: out of scope"),
)
QUESTIONS = DAY03_PROBES + DAY01_QUESTIONS
ANSWERABLE = sum(bool(p.evidence) for p in QUESTIONS)


def holds_all(evidence, text: str) -> bool:
    return bool(evidence) and all(e in text for e in evidence)


def evaluate(search, questions=QUESTIONS, texts: dict[str, str] | None = None, holds=holds_all) -> list[dict]:
    # For each question: rank of the first top-k chunk holding ALL the evidence (None if none does),
    # the top score, and the IDs returned. `search` is any function: question -> ranked hits.
    texts = {c.chunk_id: c.text for c in chunks} if texts is None else texts
    results = []
    for probe in questions:
        hits = search(probe.question)
        complete = [h.rank for h in hits if holds(probe.evidence, texts[h.id])]
        results.append({"rank": complete[0] if complete else None, "top score": hits[0].score if hits else None,
                        "hits": [h.id for h in hits]})
    return results


def summary(results: list[dict], questions=QUESTIONS) -> tuple[int, int]:
    # (answerable questions with the evidence in the top k, of which at rank 1)
    ranks = [r["rank"] for r, p in zip(results, questions) if p.evidence]
    return sum(r is not None for r in ranks), sum(r == 1 for r in ranks)


RETRIEVERS = {"BM25": lambda q: bm25.search(q, k=TOP_K), "embeddings": dense_search}
runs = {name: evaluate(search) for name, search in RETRIEVERS.items()}


def cell(result: dict, probe: Probe) -> str:
    if not probe.evidence:   # out of scope: show what it returned instead
        return "0 hits" if result["top score"] is None else f"({result['top score']:.2f})"
    return str(result["rank"]) if result["rank"] else "-"


print(f"Rank of the first chunk holding the complete evidence ('-' = not in the top {TOP_K}; "
      "out-of-scope rows show the top score)\n")
print(f"{'question':<55}{'BM25':>6}{'embeddings':>12}   probes")
for i, probe in enumerate(QUESTIONS):
    print(f"{probe.question:<55}{cell(runs['BM25'][i], probe):>6}{cell(runs['embeddings'][i], probe):>12}   {probe.probes}")
print()
for name, results in runs.items():
    found, first = summary(results)
    print(f"{name:<11} {found}/{ANSWERABLE} answerable questions found in the top {TOP_K}, {first} at rank 1")

either = sum(p.evidence != () and (b["rank"] is not None or e["rank"] is not None)
             for p, b, e in zip(QUESTIONS, runs["BM25"], runs["embeddings"]))
print(f"found by at least one of the two: {either}/{ANSWERABLE}")

Rank of the first chunk holding the complete evidence ('-' = not in the top 3; out-of-scope rows show the top score)

question                                                 BM25  embeddings   probes
What is the hotel limit per night in Singapore?             1           1   PDF table: row + column header
What is the daily meal limit in New York?                   1           1   PDF table: row + column header
Who approves admin rights?                                  1           1   HTML table: row + column header
How long is parental leave?                                 3           2   Day 01: answer was at rank 3
How many failed sign-in attempts lock my account?           1           1   one sentence
How long are application logs kept?                         1           1   one sentence
How much is the home office allowance?                      1           1   one sentence
How long do I have to claim the home office allowance?      1           1   second sentence of a section


**A tie, but not the same 14.** Each retriever finds the evidence for 14 of the 15 answerable questions, 12 of them at rank 1, and each misses a different one:

- **BM25 misses "new dads"**: no shared words (section 7.1). Embeddings find it, at rank 3.
- **Embeddings miss "lost laptop"**: the evidence is in *How to report* ("within one hour"), a section that never mentions laptops. The question's vector is closer to the sections that do (what to do with an infected laptop, reporting a lost laptop while travelling). BM25 finds it at rank 3, because that section repeats the word "report".
- Both put *"How long is parental leave?"* below rank 1 (BM25 3, embeddings 2), and both find all three table questions at rank 1: Day 03's structure-aware chunks keep each table with its header, whichever retriever reads them.

Because their misses don't overlap, every answerable question is in at least one of the two top-3 lists. That is the argument for **hybrid search**, and Day 08 tests whether combining the two into a single ranking keeps that advantage. The 17 questions were chosen to probe failures, so read the counts as "which failure types each has", not as accuracy (Day 07 builds a proper labelled set).

## 7. Where each one fails

### 7.1 Different words for the same thing: BM25's blind spot

In [15]:
for hit in bm25.search(QUESTION, k=TOP_K):
    print(f"BM25 #{hit.rank} [{hit.id}] matched only {sorted(hit.contributions)}")

BM25 #1 [security-incident-reporting#what-not-to-do] matched only ['off']
BM25 #2 [remote-work-policy#purpose] matched only ['time']
BM25 #3 [new-starter-it-checklist#your-first-day] matched only ['new']


BM25 can only reward shared words, so it ranks chunks that happen to contain "off" (*"do not turn it off"*), "time" and "new". Day 01's abstention threshold turned this into an honest "I don't know"; embeddings turn it into the right document. Stemming or synonym lists would help BM25 a little; query rewriting (Day 10) is another fix.

### 7.2 No such thing as "no match": the embeddings' blind spot

BM25 returns nothing when no chunk shares a word with the question: that is how Day 01 abstained on *"What is the capital of France?"*. An embedding search **always** returns a top 3, because every chunk has *some* cosine similarity with every question. To say "I don't know", a dense retriever needs a **threshold**: a minimum score below which results are ignored. Is there one that separates our answerable questions from the two out-of-scope ones?

In [16]:
def top_scores(answerable: bool) -> list[tuple[float, str]]:
    return sorted((r["top score"], p.question) for r, p in zip(runs["embeddings"], QUESTIONS)
                  if bool(p.evidence) == answerable)


dense_in, dense_out = top_scores(True), top_scores(False)
print("embeddings, top-1 cosine similarity (3 decimals: some scores are close)")
print(f"  {ANSWERABLE} answerable questions: {' '.join(f'{s:.3f}' for s, _ in dense_in)}")
print(f"  2 out of scope:          {'   '.join(f'{s:.3f} ({q})' for s, q in dense_out)}")

highest_out = dense_out[-1][0]
lost = [(score, q) for score, q in dense_in if score <= highest_out]
print(f"\nA threshold above {highest_out:.3f} rejects both out-of-scope questions, "
      f"and also {len(lost)} of the {ANSWERABLE} answerable ones:")
for score, q in lost:
    print(f"   {score:.3f}  {q}")

embeddings, top-1 cosine similarity (3 decimals: some scores are close)
  15 answerable questions: 0.626 0.667 0.672 0.676 0.708 0.721 0.739 0.746 0.749 0.758 0.761 0.769 0.815 0.824 0.856
  2 out of scope:          0.425 (What is the capital of France?)   0.710 (What is Google's remote work policy?)

A threshold above 0.710 rejects both out-of-scope questions, and also 5 of the 15 answerable ones:
   0.626  What training must new starters complete?
   0.667  Can I use my personal computer for work?
   0.672  Who approves admin rights?
   0.676  How soon must I report a lost laptop?
   0.708  What is the hotel limit per night in Singapore?


No threshold works for both. *"What is the capital of France?"* scores 0.425, well below every answerable question (0.626 and up), so a threshold around 0.5 would catch it. But *"What is Google's remote work policy?"* scores 0.710, higher than five answerable questions, including the Singapore hotel limit (0.708). The question is *about* a topic the documents cover (remote work), just not about this company. BM25 has the same problem: it scores the Google question 6.21, above Day 01's threshold of 4.0, which is why Day 01 recorded it as a failure.

A similarity score measures "is about the same thing", not "answers the question". Catching the Google question needs a step that reads the evidence and checks whether it answers (Day 06) and test questions like it in the evaluation set (Days 07 and 12).

### 7.3 Exact values and identifiers: BM25's home ground

People also search for exact strings: an amount, a deadline, an email address, a URL. Here the characters matter more than the meaning. To avoid picking examples that suit one side, the next cell takes **every** such string in the documents and uses each one, as is, as a query:

- every number written in digits, with its unit (`USD 220`, `13 months`, `15:00`, `1 April`; list numbering excluded), and a check that none is missed;
- every email address, URL and chat channel.

A hit counts only if the chunk contains the string as a whole: `USD 25` must not count inside `USD 250`.

In [17]:
MONTHS = "January|February|March|April|May|June|July|August|September|October|November|December"
VALUE = re.compile(rf"USD \d[\d,]*\d|\d+:\d\d|\d+ (?:business days|days|weeks|months|minutes|characters|{MONTHS})")
IDENTIFIER = re.compile(r"[\w.-]+@[\w.-]+\.\w+|https?://\S+[\w/]|#[a-z][\w-]*")

values = sorted({m for d in documents for m in VALUE.findall(d.body)})
identifiers = sorted({m for d in documents for m in IDENTIFIER.findall(d.body)})

# Completeness: every digit in the documents is inside a value or identifier, or numbers a list item
for d in documents:
    covered = [m.span() for pattern in (VALUE, IDENTIFIER) for m in pattern.finditer(d.body)]
    for line_start, line in ((m.start(), m.group()) for m in re.finditer(r"[^\n]+", d.body)):
        for digit in re.finditer(r"\d", line):
            at = line_start + digit.start()
            assert any(s <= at < e for s, e in covered) or re.match(r"\d+\. ", line), (d.doc_id, line)


def holds_whole(evidence, text: str) -> bool:
    # The whole string, not part of a longer number or word ("USD 25" must not match "USD 250").
    return all(re.search(rf"(?<!\w){re.escape(e)}(?!\w|[,.]\d)", text) for e in evidence)


LOOKUPS = {"values": tuple(Probe(v, (v,), "value") for v in values),
           "identifiers": tuple(Probe(s, (s,), "identifier") for s in identifiers)}
lookup_runs = {(kind, name): evaluate(search, probes, holds=holds_whole)
               for kind, probes in LOOKUPS.items() for name, search in RETRIEVERS.items()}

print(f"{'':<16}{'BM25':<24}embeddings")
for kind, probes in LOOKUPS.items():
    row = [summary(lookup_runs[kind, name], probes) for name in RETRIEVERS]
    print((f"{len(probes):>2} {kind:<13}" + "".join(f"{f'{f}/{len(probes)} ({r} at rank 1)':<24}" for f, r in row)).rstrip())
missed = [p.question for p, r in zip(LOOKUPS["values"], lookup_runs["values", "embeddings"]) if r["rank"] is None]
print(f"\nvalues the embeddings miss in the top {TOP_K}: {', '.join(missed)}\n")
show_hits("USD 220")

                BM25                    embeddings
25 values       25/25 (25 at rank 1)    14/25 (8 at rank 1)
 5 identifiers  5/5 (5 at rank 1)       5/5 (4 at rank 1)

values the embeddings miss in the top 3: 14 days, 15 minutes, 3 business days, 30 days, 5 business days, 60 days, USD 180, USD 220, USD 250, USD 300, USD 60

USD 220
  BM25
    #1  5.95  [travel-policy#daily-limits-by-city]  Hotel and meal costs are reimbursed up to the daily limi...
    #2  2.45  [expense-reimbursement-policy#meals]  Meals during business travel are reimbursed up to USD 75...
    #3  2.17  [expense-reimbursement-policy#approval-and-payment]  Your manager approves claims under USD 1,000; claims abo...
  embeddings
    #1  0.58  [expense-reimbursement-policy#meals]  Meals during business travel are reimbursed up to USD 75...
    #2  0.55  [remote-work-policy#equipment-and-home-office]  Employees approved for remote work receive a one-time ho...
    #3  0.55  [expense-reimbursement-policy#approval-and-pa

Exact values are where BM25 shines and embeddings struggle. BM25 puts the right chunk first for all 25 values; embeddings find 14 of 25 in the top 3. For `USD 220` (the Singapore hotel limit) they return three chunks that mention *other* dollar amounts (USD 75, USD 500, USD 1,000) and not the travel table: to this model, one dollar amount is much like another. The identifiers came out nearly even (5 of 5 found by both, one at rank 2 for embeddings), but our corpus is tiny and each address appears in one short chunk. Research on larger collections found that dense retrievers struggle with questions about rare entities, such as uncommon names ([Sciavolino et al., 2021](https://arxiv.org/abs/2109.08535)); exact codes and numbers are a related weak spot, and one reason keyword search is still part of most production search stacks.

## 8. Re-measuring two knobs: context headers and the query instruction

Day 03 found that the `Title > Section` header line helped BM25 on two questions and hurt it on one, and promised to measure again with embeddings. The model card adds a second knob: an instruction in front of queries, which it calls optional for this model version. Six variants, the same 15 answerable questions:

In [18]:
headerless = [c for d in documents for c in section_chunks(d, context_header=False)]
assert [c.chunk_id for c in headerless] == chunk_ids     # same chunks, same order; only the header line differs
headerless_texts = {c.chunk_id: c.text for c in headerless}
headerless_vectors = embed([c.text for c in headerless])
bm25_headerless = BM25Index(headerless_texts)

VARIANTS = {  # name: (search function, chunk texts used for the evidence check; None = with headers)
    "BM25, with headers": (RETRIEVERS["BM25"], None),
    "BM25, no headers": (lambda q: bm25_headerless.search(q, k=TOP_K), headerless_texts),
    "embeddings, with headers": (dense_search, None),
    "embeddings, no headers": (lambda q: dense_search(q, headerless_vectors), headerless_texts),
    "embeddings, with headers, no instruction": (lambda q: dense_search(q, instruction=""), None),
    "embeddings, no headers, no instruction": (lambda q: dense_search(q, headerless_vectors, instruction=""),
                                               headerless_texts),
}
knob_runs = {name: evaluate(search, texts=texts) for name, (search, texts) in VARIANTS.items()}

print(f"{'':<42}{'found in top 3':>15}{'at rank 1':>11}")
for name, results in knob_runs.items():
    found, first = summary(results)
    print(f"{name:<42}{found:>12}/{ANSWERABLE}{first:>11}")

for retriever in ("BM25", "embeddings"):
    print(f"\n{retriever}: questions whose rank changes when the header line is added")
    for probe, without, with_header in zip(QUESTIONS, knob_runs[f"{retriever}, no headers"],
                                           knob_runs[f"{retriever}, with headers"]):
        if probe.evidence and without["rank"] != with_header["rank"]:
            print(f"  rank {without['rank'] or '-'} -> {with_header['rank'] or '-'}   {probe.question}")

                                           found in top 3  at rank 1
BM25, with headers                                  14/15         12
BM25, no headers                                    13/15          9
embeddings, with headers                            14/15         12
embeddings, no headers                              15/15         13
embeddings, with headers, no instruction            14/15         12
embeddings, no headers, no instruction              15/15         12

BM25: questions whose rank changes when the header line is added
  rank 2 -> 3   How long is parental leave?
  rank 2 -> 1   Can I install software from the internet myself?
  rank - -> 1   What happens after I report a security incident?
  rank 2 -> 1   My account is locked, what now?

embeddings: questions whose rank changes when the header line is added
  rank 3 -> -   How soon must I report a lost laptop?
  rank 1 -> 3   How much time off do new dads get?


**The same knob moves the two retrievers in opposite directions.** For BM25 the header line helps: 14 questions found instead of 13, and 12 at rank 1 instead of 9, because a header adds the topic words that a section's text leaves out (Day 03 showed the same). For embeddings it hurts: without headers they find all 15, and *"new dads"* moves from rank 3 to rank 1.

Why? The next cell measures how similar chunks of the **same** document are to each other, with and without their headers:

In [19]:
def mean_similarity(vectors: np.ndarray) -> tuple[float, float]:
    # Average cosine between chunks of the same document, and between chunks of different documents.
    sim, docs = vectors @ vectors.T, [c.doc_id for c in chunks]
    pairs = [(i, j) for i in range(len(docs)) for j in range(i + 1, len(docs))]
    same = [sim[i, j] for i, j in pairs if docs[i] == docs[j]]
    other = [sim[i, j] for i, j in pairs if docs[i] != docs[j]]
    return float(np.mean(same)), float(np.mean(other))


within = {}
for label, vectors in [("with headers", chunk_vectors), ("no headers", headerless_vectors)]:
    within[label] = mean_similarity(vectors)
    print(f"{label:<13} same document {within[label][0]:.2f} | different documents {within[label][1]:.2f}")

q = embed_query(QUESTION)
print(f"\n'{QUESTION}' vs the parental leave chunks:")
for c, with_h, without in zip(chunks, chunk_vectors, headerless_vectors):
    if c.doc_id == "parental-leave-policy":
        print(f"  {c.chunk_id.split('#')[1]:<22} with header {with_h @ q:.2f}   without {without @ q:.2f}")

with headers  same document 0.71 | different documents 0.52
no headers    same document 0.65 | different documents 0.52

'How much time off do new dads get?' vs the parental leave chunks:
  who-is-eligible        with header 0.67   without 0.65
  leave-entitlement      with header 0.66   without 0.66
  how-to-request-leave   with header 0.62   without 0.53
  returning-to-work      with header 0.72   without 0.61


The header line makes every chunk of a document start with the same words, so their vectors move closer together (0.71 instead of 0.65 on average), while chunks of different documents stay as far apart as before (0.52). One likely reading: the vector now says more about *which document* a chunk is from and less about *what its section says*. For the "new dads" question, the header lifts *Returning to work* to first place; without headers, the section that actually states the leave lengths comes first. Day 03 saw the same effect in BM25, on the parental leave question.

The query instruction made a small difference: one more question at rank 1 without headers, none with them. We keep it, as the model card recommends.

**What we do with this.** We keep the header in the chunk text for now: BM25 needs it, readers and citations benefit from it, and one question out of 15 is a hint, not a verdict. But the finding changes how we think about the index: **the text a retriever indexes doesn't have to be the text a reader sees**. Day 08 (hybrid search) can give each retriever its own text, and Day 07's labelled questions will tell us whether the difference holds up.

## 9. Cost, and what to store with every vector

### 9.1 Time and memory, measured on this machine

Your numbers will differ: they depend on the CPU, the number of cores and whatever else is running.

In [20]:
def median_ms(search, repeats: int = 3) -> float:
    # Median milliseconds per question, over all 17 questions, repeated.
    times = []
    for _ in range(repeats):
        for probe in QUESTIONS:
            started = time.perf_counter()
            search(probe.question)
            times.append((time.perf_counter() - started) * 1000)
    return statistics.median(times)


started = time.perf_counter()
BM25Index({c.chunk_id: c.text for c in chunks})
bm25_index_ms = (time.perf_counter() - started) * 1000
query_ms = {name: median_ms(search) for name, search in RETRIEVERS.items()}

print(f"{'':<12}{'build index':>13}{'per question':>14}")
print(f"{'BM25':<12}{bm25_index_ms:>10.1f} ms{query_ms['BM25']:>11.2f} ms")
print(f"{'embeddings':<12}{index_seconds * 1000:>10.0f} ms{query_ms['embeddings']:>11.2f} ms")
print(f"\nvectors: {len(chunks)} x 384 float32 = {chunk_vectors.nbytes:,} bytes; "
      f"a million chunks would need {1_000_000 * 384 * 4 / 1e9:.1f} GB before any index")
print(f"model file: {MODEL['files']['onnx/model.onnx'][1] / 1e6:.0f} MB")

              build index  per question
BM25               2.0 ms       0.03 ms
embeddings        1112 ms      10.99 ms

vectors: 44 x 384 float32 = 67,584 bytes; a million chunks would need 1.5 GB before any index
model file: 133 MB


On the machine that produced the saved outputs, embeddings were hundreds of times slower than BM25 per question, because every question goes through the model first. At this scale both are instant for a person; at millions of chunks and many users the difference becomes hardware cost. The vectors themselves are small here, but they grow linearly: 1.5 GB per million chunks for this model, more for models with more dimensions.

### 9.2 A vector is only meaningful next to its model

Embeddings from two different models, or from two versions of one model, live in different spaces. Compare a question embedded by model B with chunks embedded by model A and you get confident-looking nonsense, with no error. So every stored vector needs to carry **which model made it** and **which text it came from**:

- `model` and `revision`: change either, and *every* chunk must be re-embedded;
- `query_instruction`, `pooling` and `normalised`: how a query must be embedded to be comparable;
- `chunk_id` and `chunk_hash`: when a document changes, only chunks whose hash changed need new vectors (Day 03 showed that one edit changes 1 of 5 structure-aware chunks).

The cache below reuses a vector whenever the model revision and the chunk hash are unchanged. Repeat Day 03's edit (one sentence added to the travel policy's *Purpose* section) and count how many chunks actually go through the model:

In [21]:
vector_cache: dict[tuple[str, str], np.ndarray] = {}   # (model revision, chunk hash) -> vector


def embed_chunks(chunk_list) -> tuple[np.ndarray, int]:
    # Embed only chunks whose (revision, hash) we haven't seen; return all vectors and how many were new.
    todo = [c for c in chunk_list if (MODEL["revision"], c.chunk_hash) not in vector_cache]
    for c, v in zip(todo, embed([c.text for c in todo]) if todo else []):
        vector_cache[MODEL["revision"], c.chunk_hash] = v
    return np.vstack([vector_cache[MODEL["revision"], c.chunk_hash] for c in chunk_list]), len(todo)


_, first_run = embed_chunks(chunks)
travel = next(d for d in documents if d.doc_id == "travel-policy")
edited = replace(travel, body=travel.body.replace("## Purpose\n", "## Purpose\nThis policy was updated in October 2026.\n"))
edited_chunks = [c for d in documents for c in section_chunks(edited if d.doc_id == "travel-policy" else d)]
_, second_run = embed_chunks(edited_chunks)
print(f"first run: {first_run} chunks embedded; after the edit: {second_run} of {len(edited_chunks)} re-embedded")

first run: 44 chunks embedded; after the edit: 1 of 44 re-embedded


Finally, save the vectors with a small manifest for Day 05, which builds a searchable index from them. `outputs/` is in `.gitignore`.

In [22]:
OUT_DIR = REPO_ROOT / "outputs" / "day-04"
OUT_DIR.mkdir(parents=True, exist_ok=True)
np.save(OUT_DIR / "chunk_vectors.npy", chunk_vectors)
manifest = {
    "model": MODEL["repo"], "revision": MODEL["revision"], "licence": MODEL["licence"],
    "dimensions": int(chunk_vectors.shape[1]), "pooling": "CLS", "normalised": True,
    "similarity": "cosine (= dot product, vectors have length 1)", "query_instruction": QUERY_INSTRUCTION,
    "embedded_text": "chunk text, including its 'Title > Section' header line",
    "chunks": [{"chunk_id": c.chunk_id, "chunk_hash": c.chunk_hash, "allowed_groups": list(c.allowed_groups)}
               for c in chunks],
}
(OUT_DIR / "chunk_vectors.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8", newline="\n")
print(f"wrote {len(chunks)} vectors to outputs/day-04/chunk_vectors.npy and their manifest to chunk_vectors.json")

wrote 44 vectors to outputs/day-04/chunk_vectors.npy and their manifest to chunk_vectors.json


Note `allowed_groups` in the manifest. A vector is computed from the text, and research has shown that text can be largely reconstructed from its embedding ([Morris et al., 2023](https://arxiv.org/abs/2310.06816)). Protect stored vectors like the documents they came from, and keep the permissions next to them: Day 05 filters on them, Day 14 enforces them.

## 10. Sanity checks

These assertions turn the behaviour above into quick, deterministic tests. `tests/test_day_04_notebook.py` runs this notebook in a fresh kernel, so any failing check fails the test. Scores can differ in the last decimal places between CPUs and operating systems, so the checks compare ranks, counts and values with a tolerance, never exact floats.

In [23]:
def close(x: float, expected: float, tol: float = 0.01) -> bool:
    return abs(x - expected) <= tol


# Input: Day 03's 44 chunks, embedded as 44 unit-length vectors of 384 numbers
assert len(documents) == 11 and len(chunks) == 44 and len(set(chunk_ids)) == 44
assert chunk_vectors.shape == (44, 384) and chunk_vectors.dtype == np.float32
assert np.allclose(np.linalg.norm(chunk_vectors, axis=1), 1, atol=1e-5)

# Tokens, padding and similarity maths (section 3 and 4)
assert encoding.tokens[0] == "[CLS]" and encoding.tokens[-1] == "[SEP]" and "##s" in encoding.tokens
assert np.abs(alone - in_batch).max() < 1e-4, "padding changed a vector: is the attention mask used?"
assert (batch_inputs["attention_mask"] == 1).all() and close(alone @ unmasked, 0.70)   # section 3.4's demo
assert (model_inputs(tokenizer.encode_batch([QUESTION, LONGER]))["attention_mask"][0] == 0).sum() == 71
assert abs(cosine - a @ b) < 1e-5 and abs(np.linalg.norm(a - b) ** 2 - (2 - 2 * cosine)) < 1e-4
assert np.argmax(similarity[0, 1:]) == 0 and close(similarity[0, 1], 0.62)        # section 4's numbers
assert close(min(unrelated), 0.30) and close(max(unrelated), 0.50)

# Head to head on 17 questions (section 6)
ranks = {name: [r["rank"] for r in results] for name, results in runs.items()}
assert ranks["BM25"] == [1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3, None, 1, None, None], ranks["BM25"]
assert ranks["embeddings"] == [1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, None, 3, 1, None, None], ranks["embeddings"]
assert summary(runs["BM25"]) == summary(runs["embeddings"]) == (14, 12) and either == ANSWERABLE == 15
assert all(c.startswith("parental-leave-policy#") for c in runs["embeddings"][13]["hits"])   # "new dads"

# Out of scope (section 7.2): BM25 can return nothing, embeddings always return 3; no threshold separates them
assert runs["BM25"][16]["top score"] is None and close(runs["BM25"][15]["top score"], 6.21)
assert all(len(r["hits"]) == TOP_K for r in runs["embeddings"])
assert close(dense_out[0][0], 0.42) and close(highest_out, 0.71) and close(dense_in[0][0], 0.63)
assert len(lost) == 5 and "What is the hotel limit per night in Singapore?" in [q for _, q in lost]

# Exact lookups (section 7.3)
assert len(values) == 25 and len(identifiers) == 5
assert summary(lookup_runs["values", "BM25"], LOOKUPS["values"]) == (25, 25)
assert summary(lookup_runs["values", "embeddings"], LOOKUPS["values"]) == (14, 8) and "USD 220" in missed
assert bm25.search("USD 220")[0].id == "travel-policy#daily-limits-by-city"
assert "travel-policy#daily-limits-by-city" not in [m.id for m in dense_search("USD 220")]
assert summary(lookup_runs["identifiers", "BM25"], LOOKUPS["identifiers"]) == (5, 5)
assert summary(lookup_runs["identifiers", "embeddings"], LOOKUPS["identifiers"])[0] == 5

# Knobs (section 8)
assert {name: summary(results) for name, results in knob_runs.items()} == {
    "BM25, with headers": (14, 12), "BM25, no headers": (13, 9),
    "embeddings, with headers": (14, 12), "embeddings, no headers": (15, 13),
    "embeddings, with headers, no instruction": (14, 12), "embeddings, no headers, no instruction": (15, 12)}
assert knob_runs["embeddings, no headers"][13]["rank"] == 1                          # "new dads" at rank 1
assert close(within["with headers"][0], 0.71) and close(within["no headers"][0], 0.65)
assert close(within["with headers"][1], 0.52) and close(within["no headers"][1], 0.52)

# Updates and the manifest (section 9)
assert first_run == 44 and second_run == 1
saved = json.loads((OUT_DIR / "chunk_vectors.json").read_text(encoding="utf-8"))
assert saved["revision"] == MODEL["revision"] and len(saved["chunks"]) == 44
assert np.array_equal(np.load(OUT_DIR / "chunk_vectors.npy"), chunk_vectors)

# Determinism: the same text gives the same vector
assert np.array_equal(embed([QUESTION]), embed([QUESTION]))

print("All Day 04 sanity checks passed.")

All Day 04 sanity checks passed.


## 11. Exercises

Each takes 5–15 minutes. Restart and rerun the notebook after changing settings.

1. **Your own vocabulary gap.** Write a question that uses none of the documents' words, for example *"Can I expense a beer on a work trip?"* (the policy says *"Alcohol is not reimbursed"*). Add it as a `Probe` and compare the two retrievers with `show_hits()`. Then write one where you expect embeddings to fail.
2. **Wrong pooling.** In `embed()`, keep the inputs (`inputs = model_inputs(tokenizer.encode_batch(...))`, then `token_vectors = run_model(inputs)`) and replace the `[CLS]` vector with the average of the real tokens' vectors: `mask = inputs["attention_mask"][..., None]` and `pooled = (token_vectors * mask).sum(1) / mask.sum(1)`. Nothing crashes. What happens to section 6's table? This is why you check a model card's pooling.
3. **Mixed queries.** `USD 220` fails with embeddings. Try `"hotel limit USD 220"` and `"Singapore hotel"`. What does the result tell you about queries that mix words and exact values, and about hybrid search on Day 08?
4. **A threshold, honestly.** Pick a dense-retrieval threshold that rejects *"What is the capital of France?"*. Write three new out-of-scope questions *before* you look at their scores (for example about another company's policy, or a policy Acme doesn't have). How many does your threshold reject? Why is choosing a threshold on the same questions you test with a mistake?
5. **Swap the model.** Pin [`sentence-transformers/all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) (Apache-2.0, ONNX file included): copy a commit hash from its *Files and versions* page, and each file's SHA-256 from its file page. Read its model card: it uses *mean* pooling and no query instruction, and cuts texts at 256 tokens. Change `MODEL`, `embed()` and `QUERY_INSTRUCTION` accordingly and rerun. Which sanity checks fail, and which failure types change?

## Summary

- **Different signals, different failures.** BM25 matches words: it can't bridge *dads* and *caregivers*, but it finds every exact value (25 of 25 at rank 1). Embeddings match meaning: they bridge the vocabulary gap, but find only 14 of 25 values and miss the lost-laptop question, whose answer sits in a section that never says "laptop". On 15 answerable questions they tie at 14, with different misses, and together they cover all 15.
- **Similarity is relative.** Unrelated texts scored 0.30 to 0.50 with this model, and an on-topic but unanswerable question scored above five answerable ones. Use scores to rank, not to decide "relevant", and test any threshold on questions you didn't tune it on.
- **A vector belongs to its model.** Pin the model like a dependency (revision + SHA-256), use its pooling, normalisation and query instruction, store the model and revision with every vector, re-embed only changed chunks, and re-embed everything when the model changes. Measure knobs such as context headers per retriever: they helped BM25 and hurt embeddings here.

**Next — Day 05: local vector search and the data lifecycle.** Today's vectors go into a local index with metadata filters, persistence, updates and deletes, and we'll see why a metadata filter is not the same thing as access control.